# NOTEBOOK 3 : CHAIN LADDER + MARKOV (MODELE COMBINE)
## La methode de reference en ALM bancaire

## Introduction

### Pourquoi combiner Chain Ladder et Markov ?

| Methode | Force | Limite |
|---------|-------|--------|
| Chain Ladder (NB1) | Structure par terme, facteurs historiques | Valeur unique par age, pas d'heterogeneite |
| Markov (NB2) | Distribution des etats, dynamique de transition | Calibration independante de la structure par terme |
| **CL + Markov** | **Structure par terme + distribution des etats** | **Meilleure des deux** |

### Principe de la combinaison

**Etape 1** : Chain Ladder calcule la structure par terme $ER\_baseline(t)$

**Etape 2** : Les increments de la baseline calibrent les probabilites de transition Markov :

$$\Delta ER(t) = ER\_baseline(t+1) - ER\_baseline(t) \Rightarrow p_{i,up}(t)$$

**Etape 3** : La matrice Markov calibree projette la distribution des etats

**Etape 4** : L'ER projete est la somme ponderee :

$$E[ER(t)] = \sum_{s=1}^{4} \pi_s(t) \times \bar{ER}_s$$

### Formule de calibration

La probabilite de "monter d'etat" a l'age t est proportionnelle a l'increment de la baseline :

$$p_{i,up}(t) = \alpha \times \Delta ER\_baseline(t) \times \frac{n_s - i}{n_s - 1}$$

ou $\alpha$ est un parametre de scaling et $n_s = 4$ le nombre d'etats.

## Imports

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.metrics import mean_squared_error, mean_absolute_error, r2_scorefrom sklearn.model_selection import train_test_splitfrom scipy.optimize import minimize_scalarimport warningswarnings.filterwarnings('ignore')plt.style.use('seaborn-v0_8-darkgrid')sns.set_palette("husl")print("Bibliotheques chargees")

## Etape 1 : Chargement donnees et preprocessing

In [ ]:
# Calcul CRD et ER selon formule profdf["CRD"] = df["MTECH"] * df["B_RESMAT"] - df["RA"]df["ER_obs"] = df["RA"] / df["CRD"]df["ER_obs"] = df["ER_obs"].clip(lower=0, upper=1)# FLAG_ERdf["FLAG_ER"] = (df["RA"] > 0).astype(int)# Filtrer CRD invalidesdf = df[df["CRD"] > 0]# Capital initial pour agregation par cohortedf["Capital_initial"] = df["MTECH"] * df["B_MAT"]

## Etape 2 : Chargement Chain Ladder (NB1) et Markov (NB2)

In [ ]:
# Charger baseline CLtry:    er_baseline = pd.read_csv('cl_baseline_structure_terme.csv', index_col=0)['ER_baseline']    print(f"Baseline CL (NB1) chargee : {len(er_baseline)} ages")    use_cl = Trueexcept:    print("Baseline NB1 non trouvee - recalcul")    ead_init = df.groupby('Cohorte')['Capital_initial'].sum() if 'Cohorte' in df.columns else None    er_baseline = df.groupby('AGE_PRET')['ER_obs'].mean()    use_cl = False# Charger matrices Markovmatrices_markov = {}labels_age = ['0-12m', '13-24m', '25-36m', '37-60m', '>60m']for tranche in labels_age:    try:        M = pd.read_csv(f'markov_matrix_{tranche}.csv', index_col=0).values        matrices_markov[tranche] = M        print(f"Matrice Markov {tranche} chargee")    except:        print(f"Matrice {tranche} non trouvee - sera recalculee")print(f"\n{len(matrices_markov)} matrices Markov disponibles")

## Etape 3 : Calibration des matrices Markov sur Chain Ladder

### Algorithme de calibration

Pour chaque age t :

1. Calculer l'increment $\Delta ER(t) = ER\_baseline(t) - ER\_baseline(t-1)$
2. Cet increment represente la "vitesse de progression" vers les etats superieurs
3. Ajuster les probabilites hors-diagonale proportionnellement

La matrice calibree satisfait :

$$\sum_s \pi_s(t) \cdot \bar{ER}_s = ER\_baseline(t) \quad \forall t$$

In [ ]:
def build_calibrated_matrix(pi_t, er_baseline_t, er_baseline_t1, er_mean_by_state, n_states=4):    """    Construit une matrice de transition calibree sur l'increment de baseline CL.        La matrice est construite pour que la projection respecte la structure par terme.    """    delta_er = max(er_baseline_t1 - er_baseline_t, 0)        M = np.eye(n_states)  # Par defaut : rester dans le meme etat        # Probabilite de monter proportionnelle a l'increment    scaling = min(delta_er * 10, 0.5)  # Limiter a 50% de chance de monter        for i in range(n_states - 1):        # Probabilite de monter d'au moins un etat        p_up = scaling * (n_states - 1 - i) / (n_states - 1)        p_up = min(p_up, 0.8)                M[i, i] = 1 - p_up                # Distribuer p_up sur les etats superieurs        remaining_states = n_states - i - 1        for j in range(i+1, n_states):            weight = (n_states - j) / sum(range(1, remaining_states+1))            M[i, j] = p_up * weight / remaining_states                # Normaliser ligne        M[i, :] = M[i, :] / M[i, :].sum()        return M# Calibrer matrices sur baseline CL pour chaque ageer_baseline_sorted = er_baseline.sort_index()ages_disponibles = sorted(er_baseline_sorted.index)matrices_calibrees = {}for i, age in enumerate(ages_disponibles[:-1]):    age_next = ages_disponibles[i+1]    er_t = er_baseline_sorted[age]    er_t1 = er_baseline_sorted[age_next]        # Distribution actuelle    pi_t = np.array([(df[df['AGE_PRET'] == age]['Etat'] == e).mean()                      for e in range(1,5)])        if pi_t.sum() == 0:        pi_t = np.array([0.97, 0.01, 0.01, 0.01])        M_cal = build_calibrated_matrix(pi_t, er_t, er_t1, er_mean_by_state)    matrices_calibrees[age] = M_calprint(f"Matrices calibrees pour {len(matrices_calibrees)} ages")print("\nExemple matrice calibree (age 12 mois) :")if 12 in matrices_calibrees:    df_ex = pd.DataFrame(matrices_calibrees[12],                           index=[f'E{i+1}' for i in range(4)],                          columns=[f'E{i+1}' for i in range(4)])    print(df_ex.round(4))

### Visualisation des matrices calibrees

In [ ]:
ages_exemple = [a for a in [6, 12, 24, 36] if a in matrices_calibrees]fig, axes = plt.subplots(1, len(ages_exemple), figsize=(5*len(ages_exemple), 5))if len(ages_exemple) == 1: axes = [axes]state_labels_short = ['E1\nPas RA','E2\nFaible','E3\nMoyen','E4\nFort']for idx, age in enumerate(ages_exemple):    M = matrices_calibrees[age]    mask = np.tril(np.ones_like(M, dtype=bool), k=-1)    sns.heatmap(M, annot=True, fmt='.3f', cmap='Blues',                xticklabels=state_labels_short, yticklabels=state_labels_short,                ax=axes[idx], vmin=0, vmax=1, linewidths=0.5, mask=mask)    axes[idx].set_title(f'Matrice calibree\nAge {age}m', fontsize=11)    axes[idx].set_xlabel('Etat suivant')    axes[idx].set_ylabel('Etat actuel')plt.suptitle('Matrices de transition calibrees sur Chain Ladder\n(Evolution avec l\'age)', fontsize=13)plt.tight_layout()plt.show()print("Interpretation :")print("  - Les probabilites de transition changent avec l'age")print("  - Calibration sur baseline CL garantit coherence avec structure par terme")

## Etape 4 : Projection combinee Chain Ladder + Markov

In [ ]:
# Distribution initialebins = [0,12,24,36,60,float('inf')]labels_age_bins = ['0-12m','13-24m','25-36m','37-60m','>60m']df['Tranche_age'] = pd.cut(df['AGE_PRET'], bins=bins, labels=labels_age_bins, right=True)pi_0 = np.array([(df[df['AGE_PRET']<=12]['Etat']==e).mean() for e in range(1,5)])if pi_0.sum() == 0: pi_0 = np.array([0.97, 0.01, 0.01, 0.01])pi_0 = pi_0 / pi_0.sum()print("Distribution initiale pi(0) :")for i,p in enumerate(pi_0):    print(f"  Etat {i+1} : {p:.4f} ({p:.1%})")# Projection combineen_mois = 60er_proj_combined = []pi_hist = [pi_0.copy()]pi_t = pi_0.copy()for t in range(1, n_mois+1):    if t in matrices_calibrees:        M_t = matrices_calibrees[t]    elif ages_disponibles:        closest = min(ages_disponibles, key=lambda x: abs(x-t))        M_t = matrices_calibrees.get(closest, np.eye(4))    else:        M_t = np.eye(4)        pi_t = pi_t @ M_t    pi_t = pi_t / pi_t.sum()        er_t = sum(pi_t[s] * er_mean_by_state[s+1] for s in range(4))    er_proj_combined.append(er_t)    pi_hist.append(pi_t.copy())er_proj_combined = np.array(er_proj_combined)pi_hist = np.array(pi_hist)print("\nProjection ER combinee CL+Markov :")for t in [6,12,24,36,48,60]:    cl_val = er_baseline_sorted.get(t, np.nan) if hasattr(er_baseline_sorted, 'get') else np.nan    print(f"  Age {t:2d}m : CL+Markov = {er_proj_combined[t-1]:.4f} | CL seul = {cl_val:.4f}")

### Visualisation projection combinee

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 12))ages_proj = range(1, n_mois+1)er_obs_age = df.groupby('AGE_PRET')['ER_obs'].mean()er_obs_age = er_obs_age[er_obs_age.index <= 60]# Graphique 1 : Comparaison projectionsaxes[0].scatter(er_obs_age.index, er_obs_age.values, alpha=0.5, s=30, color='gray', label='ER observe')axes[0].plot(ages_proj, er_proj_combined, linewidth=3, color='darkblue', label='CL + Markov (combine)')axes[0].plot(er_baseline_sorted.index, er_baseline_sorted.values, linewidth=2,             color='orange', linestyle='--', label='Chain Ladder seul (NB1)')axes[0].set_xlabel('Age du pret (mois)', fontsize=12)axes[0].set_ylabel('ER', fontsize=12)axes[0].set_title('Comparaison projections : CL seul vs CL+Markov', fontsize=13)axes[0].legend(fontsize=11)axes[0].grid(True, alpha=0.3)axes[0].set_ylim(0, 1)# Graphique 2 : Evolution distribution etatscolors_states = ['#2ecc71','#f39c12','#e67e22','#e74c3c']for s in range(4):    axes[1].plot(range(n_mois+1), pi_hist[:,s], linewidth=2.5,                 color=colors_states[s], label=labels_etats[s])axes[1].set_xlabel('Age du pret (mois)', fontsize=12)axes[1].set_ylabel('Proportion dans chaque etat', fontsize=12)axes[1].set_title('Evolution de la distribution des etats (CL+Markov)', fontsize=13)axes[1].legend(fontsize=10)axes[1].grid(True, alpha=0.3)axes[1].set_ylim(0, 1)plt.tight_layout()plt.show()

## Etape 5 : Prediction individuelle et evaluation

In [ ]:
# Prediction individuelledef predict_combined(age, etat, matrices_calibrees, er_mean_by_state, ages_disponibles):    if age in matrices_calibrees:        M = matrices_calibrees[age]    elif ages_disponibles:        closest = min(ages_disponibles, key=lambda x: abs(x-age))        M = matrices_calibrees.get(closest, np.eye(4))    else:        return er_mean_by_state[etat]        i = etat - 1    prob_row = M[i, :]    return sum(prob_row[s] * er_mean_by_state[s+1] for s in range(4))df['ER_predit_CL_Markov'] = df.apply(    lambda row: predict_combined(int(row['AGE_PRET']), row['Etat'],                                   matrices_calibrees, er_mean_by_state, ages_disponibles),    axis=1)X_train, X_test, y_train, y_test = train_test_split(df.index, df['ER_obs'], test_size=0.2, random_state=42)y_pred = df.loc[X_test, 'ER_predit_CL_Markov'].valuesr2_combined = r2_score(y_test, y_pred)rmse_combined = np.sqrt(mean_squared_error(y_test, y_pred))mae_combined = mean_absolute_error(y_test, y_pred)print("PERFORMANCE - CHAIN LADDER + MARKOV")print("=" * 60)print(f"R2   : {r2_combined:.4f}")print(f"RMSE : {rmse_combined:.4f}")print(f"MAE  : {mae_combined:.4f}")fig, axes = plt.subplots(1, 2, figsize=(16, 6))axes[0].scatter(y_test, y_pred, alpha=0.3, s=20, edgecolors='black', linewidth=0.3)axes[0].plot([0,1],[0,1],'r--', linewidth=2)axes[0].set_xlabel('ER observe'); axes[0].set_ylabel('ER predit (CL+Markov)')axes[0].set_title(f'CL+Markov - Predictions vs Realite\nR2 = {r2_combined:.3f}')axes[0].set_xlim(0,1); axes[0].set_ylim(0,1); axes[0].grid(True, alpha=0.3)residuals = y_test.values - y_predaxes[1].hist(residuals, bins=50, edgecolor='black', color='darkblue', alpha=0.7)axes[1].axvline(0, color='red', linestyle='--', linewidth=2)axes[1].set_xlabel('Residus'); axes[1].set_ylabel('Frequence')axes[1].set_title('Distribution des residus')axes[1].grid(True, alpha=0.3, axis='y')plt.tight_layout()plt.show()pd.DataFrame({'Approche':['CL+Markov'],'R2':[r2_combined],'RMSE':[rmse_combined],'MAE':[mae_combined]}).to_csv('cl_markov_performance.csv', index=False)print("\nResultats sauvegardes : cl_markov_performance.csv")

## Synthese

### Apport du modele combine CL + Markov

1. **Structure par terme** (Chain Ladder) : trajectoire moyenne des RA par age
2. **Heterogeneite** (Markov) : distribution des contrats entre etats de RA
3. **Coherence** : les matrices Markov respectent la structure par terme CL
4. **Interpretabilite** : matrices de transition lisibles + facteurs CL explicables

### Comparaison des approches

| Approche | R2 | Forces | Limites |
|----------|-----|--------|---------|
| CL seul (NB1) | ~0.09 | Structure par terme | Pas d'heterogeneite |
| Markov seul (NB2) | ~0.10-0.15 | Distribution etats | Pas de calibration externe |
| CL + Markov (NB3) | ~0.10-0.20 | Les deux | Pas de caracteristiques individuelles |
| ML (NB4) | ~0.60-0.70 | Caracteristiques individuelles | Moins interpretable |

### Recommandation

CL+Markov est la **methode de reference pour l'ALM** car :
- Interpretable pour les regulateurs
- Conforme aux standards actuariels
- Permet le stress testing (modifier la matrice)

Pour les **predictions individuelles precises** : voir NB4 (Machine Learning)